### Tools

Model request to call tools that can perform some tasks. Tools are pairing of  a schema (name, description, argument definition) and function to execute

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [4]:
model = init_chat_model("groq:qwen/qwen3-32b")

In [8]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather for a location"""
    return f"It's rainy in {location}"

In [9]:
model_with_tools = model.bind_tools([get_weather])

In [10]:
model_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F5E83656A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F5E83667B0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather for a location', 'p

In [11]:
response = model_with_tools.invoke("What is the weather in chennai ?")
print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Chennai. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Chennai, I need to call this function with "Chennai" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': 'frv5hnzcv', 'function': {'arguments': '{"location":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 154, 'total_tokens': 260, 'completion_time': 0.173418046, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.017856585, 'prompt_tokens_details': None, 'queue_time': 0.080315954, 'total_time': 0.191274631}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_r

In [12]:
for tool_call in response.tool_calls:
    print(f"tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

tool: get_weather
Args: {'location': 'Chennai'}


#### Tool Execution Loops

In [13]:
# step 1: model generate tool calls
messages = [{"role": "user", "content": "What is the weather in chennai"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

In [14]:
# step 2: execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

In [16]:
# step 3: pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Chennai is rainy.


In [17]:
messages

[{'role': 'user', 'content': 'What is the weather in chennai'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Chennai. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Chennai, I need to call this function with "Chennai" as the location. I should make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'whhc3eh89', 'function': {'arguments': '{"location":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 153, 'total_tokens': 251, 'completion_time': 0.155773578, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.006126492, 'prompt_tokens_details': None, 'queue_time': 0.054066908, 'total_time': 0.16190007}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921ca

In [18]:
get_weather

StructuredTool(name='get_weather', description='Get the weather for a location', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001F5E85011C0>)